# ProfitPilot Kaggle P100 5m Multi-Asset PPO Training

Built for **Kaggle Notebooks with a P100 GPU accelerator**.

It downloads BTC/USDT and ETH/USDT 5-minute candles from `2024-01-01` through `2026-04-28`,
trains the RL trading model on the `2024-01-01` to `2026-03-31` portion,
and tests out of sample on `2026-04-01` through `2026-04-28`.

### Kaggle setup
1. Upload the ProfitPilot project folder as a **Kaggle Dataset** (slug `profitpilot` works best).
2. In *Notebook settings → Data*, add that dataset — it will appear at `/kaggle/input/profitpilot/`.
3. Set the **Accelerator** to *GPU P100* and enable *Internet*.
4. Alternatively, set a Kaggle **Secret** named `PROFIT_PILOT_ROOT` pointing to the mounted path.

All outputs (model, reports, data) are written to `/kaggle/working/` and can be downloaded or saved as a dataset.

In [1]:
# Install project dependencies without overwriting Kaggle's pre-installed CUDA PyTorch build.
%pip install -q ccxt ta stable-baselines3 sb3-contrib gymnasium pyarrow seaborn python-dotenv PyYAML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.3/143.3 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 76.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 17.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys
print(os.getcwd())
print(sys.executable)


/kaggle/working
/usr/bin/python3


In [3]:
import os
print(os.listdir(".")[:20])


['.virtual_documents']


In [4]:
import sys
sys.path.append("src")


In [5]:
from pathlib import Path
import json
import math
import os
import random
import subprocess
import sys
import warnings

# ---------------------------------------------------------------------------
# Project root detection — robust Kaggle-first search
# ---------------------------------------------------------------------------

def _find_profit_pilot_root() -> Path | None:
    """Search for the ProfitPilot project root that contains src/profit_pilot."""

    def is_valid_root(p: Path) -> bool:
        try:
            return (p / 'src' / 'profit_pilot').exists()
        except OSError:
            return False

    # 1. Explicit env variable / Kaggle secret takes priority
    env_root = os.environ.get('PROFIT_PILOT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if is_valid_root(candidate):
            return candidate

    # 2. Scan every top-level directory under /kaggle/input/ (handles any slug
    #    and one level of nesting, e.g. /kaggle/input/profitpilot/ProfitPilot/)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for top in sorted(kaggle_input.iterdir()):
            if is_valid_root(top):
                return top.resolve()
            # one sublevel deeper (folder-inside-dataset uploads)
            try:
                for sub in sorted(top.iterdir()):
                    if sub.is_dir() and is_valid_root(sub):
                        return sub.resolve()
            except PermissionError:
                pass

    # 3. Walk up from cwd (works for local / Colab use)
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        try:
            if is_valid_root(candidate.resolve()):
                return candidate.resolve()
        except OSError:
            pass

    return None


PROJECT_ROOT = _find_profit_pilot_root()

if PROJECT_ROOT is None:
    # Print what is actually available so the user can diagnose the slug mismatch
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        available = [str(p) for p in sorted(kaggle_input.iterdir())]
        print('Available Kaggle input paths:')
        for p in available:
            print(f'  {p}')
    raise FileNotFoundError(
        'Could not find ProfitPilot (src/profit_pilot not found anywhere under /kaggle/input/). '
        'Make sure you:\n'
        '  1. Uploaded the ProfitPilot project folder as a Kaggle Dataset.\n'
        '  2. Added that dataset to this notebook (Notebook settings → Data).\n'
        '  3. Or set a Kaggle Secret named PROFIT_PILOT_ROOT with the correct path.'
    )

# Kaggle /kaggle/input/ is read-only — write outputs to /kaggle/working/
WORKING_ROOT = Path(os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working'))
if not WORKING_ROOT.exists():
    WORKING_ROOT = PROJECT_ROOT  # fallback for non-Kaggle environments

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Project root : {PROJECT_ROOT}')
print(f'Working root : {WORKING_ROOT}')

import numpy as np
import pandas as pd
import torch
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {DEVICE}')
if DEVICE == 'cuda':
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('GPU not available; training will run on CPU. Enable a P100 accelerator in Notebook settings.')


Available Kaggle input paths:


FileNotFoundError: Could not find ProfitPilot (src/profit_pilot not found anywhere under /kaggle/input/). Make sure you:
  1. Uploaded the ProfitPilot project folder as a Kaggle Dataset.
  2. Added that dataset to this notebook (Notebook settings → Data).
  3. Or set a Kaggle Secret named PROFIT_PILOT_ROOT with the correct path.

In [ ]:
import importlib
import inspect

import ccxt
import matplotlib.pyplot as plt
import seaborn as sns
from sb3_contrib import RecurrentPPO
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

import profit_pilot.data.download_ohlcv as download_ohlcv_module
import profit_pilot.env.multi_crypto_env as multi_crypto_env_module
import profit_pilot.features.build_features as build_features_module
import profit_pilot.utils.io as io_module

# Reload project modules so stale cached versions are replaced.
download_ohlcv_module = importlib.reload(download_ohlcv_module)
multi_crypto_env_module = importlib.reload(multi_crypto_env_module)
build_features_module = importlib.reload(build_features_module)
io_module = importlib.reload(io_module)

fetch_symbol_ohlcv = download_ohlcv_module.fetch_symbol_ohlcv
save_symbol_csv = download_ohlcv_module.save_symbol_csv
MultiCryptoTradingEnv = multi_crypto_env_module.MultiCryptoTradingEnv
BASE_FEATURE_COLUMNS = build_features_module.BASE_FEATURE_COLUMNS
add_technical_indicators = build_features_module.add_technical_indicators
align_frames = build_features_module.align_frames
build_arrays = build_features_module.build_arrays
bundle_directory = io_module.bundle_directory
save_json = io_module.save_json
symbol_slug = io_module.symbol_slug

if 'requested_trade_value' not in inspect.getsource(MultiCryptoTradingEnv._apply_buy_actions):
    raise RuntimeError(
        'Loaded an old MultiCryptoTradingEnv without notional multi-asset actions. '
        'Restart the runtime and make sure the latest src/profit_pilot/env/multi_crypto_env.py is present.'
    )
if 'cash_target_allocation' not in inspect.getsource(MultiCryptoTradingEnv._validate_settings):
    raise RuntimeError(
        'Loaded an old MultiCryptoTradingEnv without explicit cash allocation actions. '
        'Restart the runtime and rerun the setup/import cells.'
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_random_seed(SEED)

# Exchange fallback order — Binance is blocked in some regions (HTTP 451).
EXCHANGE_IDS = [
    exchange_id.strip()
    for exchange_id in os.environ.get('PROFIT_PILOT_EXCHANGES', 'binanceus,kucoin,okx,bybit,binance').split(',')
    if exchange_id.strip()
]
EXCHANGE_ID = EXCHANGE_IDS[0]
SYMBOLS = ['BTC/USDT', 'ETH/USDT']
TIMEFRAME = '5m'
DOWNLOAD_LIMIT = 1000

DATA_START_ISO = '2024-01-01T00:00:00Z'
TEST_START_ISO  = '2026-04-01T00:00:00Z'
DATA_END_ISO    = '2026-04-28T23:55:00Z'

INITIAL_CASH = 10_000.0
LOOKBACK = 96
# P100 has the same 16 GB VRAM as T4; 1.5 M steps is a good default.
TOTAL_TIMESTEPS = int(os.environ.get('PROFIT_PILOT_TOTAL_TIMESTEPS', '1500000'))
TRAIN_MODEL = True
USE_LSTM = True
EVALUATION_DETERMINISTIC = True
FORCE_REDOWNLOAD = False
RUN_FINAL_FULL_WINDOW_RETRAIN = False

# Kaggle P100 can fail in cuDNN's LSTM weight-flattening path with
# cudaErrorNoKernelImageForDevice. Keep CUDA enabled, but use PyTorch's
# non-cuDNN LSTM kernels for RecurrentPPO on P100.
if USE_LSTM and DEVICE == 'cuda' and 'P100' in torch.cuda.get_device_name(0).upper():
    torch.backends.cudnn.enabled = False
    print('cuDNN disabled for RecurrentPPO LSTM on P100; CUDA remains enabled.')

# All writable paths live under WORKING_ROOT so Kaggle's read-only input is untouched.
RAW_ROOT             = WORKING_ROOT / 'data' / 'raw_kaggle_p100_5m_2024_2026'
PROCESSED_TRAIN_ROOT = WORKING_ROOT / 'data' / 'processed_kaggle_p100_5m_train_2024_2026_cash'
PROCESSED_TEST_ROOT  = WORKING_ROOT / 'data' / 'processed_kaggle_p100_5m_eval_2026_0401_0428_cash'
PROCESSED_FULL_ROOT  = WORKING_ROOT / 'data' / 'processed_kaggle_p100_5m_full_2024_2026_cash'
MODEL_DIR            = WORKING_ROOT / 'models' / 'kaggle_p100_5m_cash_allocation'
REPORT_DIR           = WORKING_ROOT / 'reports' / 'kaggle_p100_5m_cash_allocation'
PLOT_DIR             = REPORT_DIR / 'plots'
for path in [RAW_ROOT, PROCESSED_TRAIN_ROOT, PROCESSED_TEST_ROOT, PROCESSED_FULL_ROOT,
             MODEL_DIR, REPORT_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = f"profit_pilot_{'recurrent_' if USE_LSTM else ''}ppo_5m_p100_cash_multi_asset"
MODEL_PATH = MODEL_DIR / MODEL_NAME

print(f'Exchange fallback order : {EXCHANGE_IDS}')
print(f'Model name             : {MODEL_NAME}')
print(f'Total timesteps        : {TOTAL_TIMESTEPS:,}')

## Download and Cache 5m Market Data

In [ ]:
def create_market_data_exchange(exchange_id: str):
    exchange_class = getattr(ccxt, exchange_id)
    exchange = exchange_class({'enableRateLimit': True})
    exchange.timeout = 30_000
    return exchange


def read_cached_frame(path: Path) -> pd.DataFrame | None:
    if not path.exists() or FORCE_REDOWNLOAD:
        return None

    frame = pd.read_csv(path)
    if frame.empty or 'datetime' not in frame.columns:
        return None

    frame['datetime'] = pd.to_datetime(frame['datetime'], utc=True)
    start = pd.Timestamp(DATA_START_ISO)
    end = pd.Timestamp(DATA_END_ISO)
    if frame['datetime'].min() <= start and frame['datetime'].max() >= end - pd.Timedelta(minutes=5):
        return frame[(frame['datetime'] >= start) & (frame['datetime'] <= end)].copy()
    return None


def load_or_download_symbol(exchange, exchange_id: str, symbol: str) -> pd.DataFrame:
    target_path = RAW_ROOT / exchange_id / TIMEFRAME / f'{symbol_slug(symbol)}.csv'
    cached = read_cached_frame(target_path)
    if cached is not None:
        print(f'Using cached {symbol} from {exchange_id}: {len(cached):,} rows')
        return cached

    print(f'Downloading {symbol} {TIMEFRAME} from {exchange_id}: {DATA_START_ISO} to {DATA_END_ISO}')
    frame = fetch_symbol_ohlcv(
        exchange=exchange,
        symbol=symbol,
        timeframe=TIMEFRAME,
        since=DATA_START_ISO,
        until=DATA_END_ISO,
        limit=DOWNLOAD_LIMIT,
    )
    frame['datetime'] = pd.to_datetime(frame['datetime'], utc=True)
    saved_path = save_symbol_csv(frame, RAW_ROOT, exchange_id, TIMEFRAME, symbol)
    print(f'Saved {len(frame):,} rows to {saved_path}')
    return frame


def download_all_symbols_from_exchange(exchange_id: str) -> dict[str, pd.DataFrame]:
    exchange = create_market_data_exchange(exchange_id)
    try:
        exchange.load_markets()
        available_symbols = set(exchange.symbols or [])
        missing_symbols = [symbol for symbol in SYMBOLS if symbol not in available_symbols]
        if missing_symbols:
            raise ValueError(f'{exchange_id} does not list these symbols: {missing_symbols}')

        return {
            symbol: load_or_download_symbol(exchange, exchange_id, symbol)
            for symbol in SYMBOLS
        }
    finally:
        if hasattr(exchange, 'close'):
            exchange.close()


full_symbol_frames = None
download_errors = {}
for candidate_exchange_id in EXCHANGE_IDS:
    print(f'\nTrying exchange: {candidate_exchange_id}')
    try:
        candidate_frames = download_all_symbols_from_exchange(candidate_exchange_id)
        full_symbol_frames = candidate_frames
        EXCHANGE_ID = candidate_exchange_id
        print(f'Using exchange for this run: {EXCHANGE_ID}')
        break
    except Exception as exc:
        error_message = f'{type(exc).__name__}: {str(exc)}'
        download_errors[candidate_exchange_id] = error_message
        print(f'{candidate_exchange_id} failed: {error_message[:500]}')

if full_symbol_frames is None:
    display(pd.DataFrame(download_errors.items(), columns=['exchange', 'error']))
    raise RuntimeError(
        'No exchange in EXCHANGE_IDS could download all requested symbols. '
        'Try changing EXCHANGE_IDS in the config cell, for example to ["kucoin", "okx", "bybit"].'
    )

coverage_rows = []
for symbol, frame in full_symbol_frames.items():
    coverage_rows.append({
        'exchange': EXCHANGE_ID,
        'symbol': symbol,
        'rows': len(frame),
        'first': frame['datetime'].min(),
        'last': frame['datetime'].max(),
    })
display(pd.DataFrame(coverage_rows))

## Build Train and Test Feature Bundles

In [ ]:
def split_symbol_frames(frames: dict[str, pd.DataFrame], start_iso: str, end_iso: str, inclusive_end: bool) -> dict[str, pd.DataFrame]:
    start = pd.Timestamp(start_iso)
    end = pd.Timestamp(end_iso)
    split_frames = {}
    for symbol, frame in frames.items():
        working = frame.copy()
        working['datetime'] = pd.to_datetime(working['datetime'], utc=True)
        if inclusive_end:
            mask = (working['datetime'] >= start) & (working['datetime'] <= end)
        else:
            mask = (working['datetime'] >= start) & (working['datetime'] < end)
        split = working.loc[mask].sort_values('datetime').reset_index(drop=True)
        if split.empty:
            raise ValueError(f'No rows for {symbol} between {start_iso} and {end_iso}')
        split_frames[symbol] = split
    return split_frames


def build_normalized_feature_bundle(
    symbol_frames: dict[str, pd.DataFrame],
    processed_root: Path,
    source_start_iso: str,
    source_end_iso: str,
    label: str,
    normalization_stats: dict[str, np.ndarray] | None = None,
) -> tuple[dict, dict[str, np.ndarray]]:
    enriched_frames = {}
    for symbol, frame in symbol_frames.items():
        enriched = add_technical_indicators(frame)
        if enriched.empty:
            raise ValueError(f'Feature frame became empty after indicator warm-up: {symbol}')
        enriched_frames[symbol] = enriched

    timestamps, aligned_frames = align_frames(enriched_frames)
    price_array, raw_tech_array, feature_export = build_arrays(SYMBOLS, aligned_frames)

    if normalization_stats is None:
        tech_mean = raw_tech_array.mean(axis=0)
        tech_std = raw_tech_array.std(axis=0)
        tech_std = np.where(tech_std < 1e-8, 1.0, tech_std)
        normalization_stats = {'mean': tech_mean, 'std': tech_std}

    tech_array = ((raw_tech_array - normalization_stats['mean']) / normalization_stats['std']).astype(np.float32)
    price_array = price_array.astype(np.float32)

    output_dir = bundle_directory(processed_root, TIMEFRAME, SYMBOLS)
    output_dir.mkdir(parents=True, exist_ok=True)
    np.save(output_dir / 'price_array.npy', price_array)
    np.save(output_dir / 'tech_array.npy', tech_array)
    np.save(output_dir / 'tech_mean.npy', normalization_stats['mean'])
    np.save(output_dir / 'tech_std.npy', normalization_stats['std'])
    pd.DataFrame({'datetime': timestamps.astype(str)}).to_csv(output_dir / 'timestamps.csv', index=False)
    feature_export.to_parquet(output_dir / 'feature_frame.parquet', index=False)

    metadata = {
        'label': label,
        'symbols': SYMBOLS,
        'feature_columns_per_asset': BASE_FEATURE_COLUMNS,
        'n_assets': len(SYMBOLS),
        'n_features_per_asset': len(BASE_FEATURE_COLUMNS),
        'price_array_shape': list(price_array.shape),
        'tech_array_shape': list(tech_array.shape),
        'timeframe': TIMEFRAME,
        'source_since': source_start_iso,
        'source_until': source_end_iso,
        'first_timestamp_after_warmup': str(timestamps.min()),
        'last_timestamp': str(timestamps.max()),
        'normalization': 'standard_score_using_train_stats',
    }
    save_json(metadata, output_dir / 'metadata.json')

    bundle = {
        'root': output_dir,
        'price_array': price_array,
        'tech_array': tech_array,
        'timestamps': pd.DataFrame({'datetime': timestamps.astype(str)}),
        'metadata': metadata,
    }
    return bundle, normalization_stats


train_frames = split_symbol_frames(full_symbol_frames, DATA_START_ISO, TEST_START_ISO, inclusive_end=False)
test_frames  = split_symbol_frames(full_symbol_frames, TEST_START_ISO,  DATA_END_ISO,  inclusive_end=True)

train_bundle, train_norm_stats = build_normalized_feature_bundle(
    train_frames,
    PROCESSED_TRAIN_ROOT,
    DATA_START_ISO,
    TEST_START_ISO,
    label='train_2024_0101_2026_0331',
)
test_bundle, _ = build_normalized_feature_bundle(
    test_frames,
    PROCESSED_TEST_ROOT,
    TEST_START_ISO,
    DATA_END_ISO,
    label='test_2026_0401_0428',
    normalization_stats=train_norm_stats,
)

bundle_summary = pd.DataFrame([train_bundle['metadata'], test_bundle['metadata']])
display(bundle_summary[[
    'label',
    'timeframe',
    'source_since',
    'source_until',
    'first_timestamp_after_warmup',
    'last_timestamp',
    'price_array_shape',
    'tech_array_shape',
]])

## Train Multi-Asset PPO on the P100 GPU

In [ ]:
ENV_KWARGS = dict(
    lookback=LOOKBACK,
    initial_cash=INITIAL_CASH,
    buy_cost_pct=0.001,
    sell_cost_pct=0.001,
    cash_norm=0.0001,
    holdings_norm=1.0,
    tech_norm=1.0,
    reward_scaling=1.0,
    reward_mode='net_log_return_alpha',
    action_mode='cash_target_allocation',
    cost_penalty_weight=0.10,
    risk_penalty_weight=0.10,
    alpha_reward_weight=1.0,
    drawdown_penalty_weight=0.20,
    risk_halt_penalty_weight=0.05,
    volatility_reward_weight=0.0,
    volatility_window=288,
    risk_adjusted_return_clip=5.0,
    max_position_fraction=0.60,
    max_gross_exposure=0.80,
    max_trade_fraction=0.50,
    stop_loss_pct=0.07,
    max_drawdown_pct=0.25,
    min_trade_quantity=0.0001,
    cooldown_steps=6,
    trade_deadband=0.10,
    rebalance_threshold=0.08,
    min_trade_notional=50.0,
    turnover_penalty_weight=0.02,
)

TRAIN_ENV_KWARGS = dict(ENV_KWARGS, random_start=True, episode_length=12 * 24 * 10)


def make_trading_env(bundle: dict, training: bool = False) -> MultiCryptoTradingEnv:
    if bundle['price_array'].shape[0] <= LOOKBACK + 2:
        raise ValueError('Not enough rows for the configured LOOKBACK.')
    env_kwargs = TRAIN_ENV_KWARGS if training else ENV_KWARGS
    return MultiCryptoTradingEnv(
        price_array=bundle['price_array'],
        tech_array=bundle['tech_array'],
        tickers=SYMBOLS,
        **env_kwargs,
    )


def make_vec_env(bundle: dict):
    def _make_env():
        return make_trading_env(bundle, training=True)

    return VecMonitor(DummyVecEnv([_make_env]))


train_env = make_vec_env(train_bundle)
print(f'Observation shape: {train_env.observation_space.shape}')
print(f'Action shape     : {train_env.action_space.shape}')

sanity_env = make_trading_env(train_bundle)
sanity_env.reset()
sanity_action = np.ones(sanity_env.action_space.shape, dtype=np.float32)
_, _, _, _, sanity_info = sanity_env.step(sanity_action)
sanity_positions = sanity_env.holdings * sanity_env.price_array[sanity_env.time]
sanity_exposures = sanity_positions / max(float(sanity_info['portfolio_value']), 1e-12)
sanity_table = pd.DataFrame({
    'symbol': SYMBOLS,
    'holding_after_positive_action': sanity_env.holdings,
    'position_value_after_positive_action': sanity_positions,
    'exposure_after_positive_action': sanity_exposures,
})
display(sanity_table)

if sanity_env.action_space.shape[0] != len(SYMBOLS) + 1:
    raise RuntimeError('Expected explicit cash allocation action plus one action per asset.')
if not np.all(sanity_positions > 1.0):
    raise RuntimeError(
        'Multi-asset sanity check failed: a positive asset action did not create notional exposure for every asset. '
        'Restart the runtime and rerun the setup/import cells.'
    )

In [ ]:
checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path=str(MODEL_DIR / 'checkpoints'),
    name_prefix=MODEL_NAME,
)

common_model_kwargs = dict(
    env=train_env,
    verbose=1,
    learning_rate=3e-5,
    n_steps=4096,
    batch_size=1024,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.08,
    ent_coef=0.001,
    target_kl=0.02,
    device=DEVICE,
    tensorboard_log=str(REPORT_DIR / 'tensorboard'),
)

if TRAIN_MODEL:
    if USE_LSTM:
        model = RecurrentPPO('MlpLstmPolicy', **common_model_kwargs)
    else:
        model = PPO('MlpPolicy', policy_kwargs={'net_arch': [256, 256]}, **common_model_kwargs)

    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=checkpoint_callback, progress_bar=True)
    model.save(MODEL_PATH)
    train_env.close()

    training_summary = {
        'model_path': str(MODEL_PATH) + '.zip',
        'model_name': MODEL_NAME,
        'use_lstm': USE_LSTM,
        'device': DEVICE,
        'total_timesteps': TOTAL_TIMESTEPS,
        'symbols': SYMBOLS,
        'timeframe': TIMEFRAME,
        'train_window': [DATA_START_ISO, TEST_START_ISO],
        'test_window': [TEST_START_ISO, DATA_END_ISO],
        'lookback': LOOKBACK,
        'train_price_array_shape': list(train_bundle['price_array'].shape),
        'train_tech_array_shape': list(train_bundle['tech_array'].shape),
        'environment': ENV_KWARGS,
        'training_hyperparameters': {
            key: value
            for key, value in common_model_kwargs.items()
            if key not in {'env'}
        },
    }
    save_json(training_summary, MODEL_DIR / f'{MODEL_NAME}_training_summary.json')
else:
    model_class = RecurrentPPO if USE_LSTM else PPO
    model = model_class.load(MODEL_PATH, device=DEVICE)
    train_env.close()

## Evaluate the Model Out of Sample

In [ ]:
def make_eval_record(env: MultiCryptoTradingEnv, timestamps: pd.Series, step: int, reward: float, action, info: dict) -> dict:
    action_dim = env.action_space.shape[0]
    action_array = np.zeros(action_dim, dtype=np.float32) if action is None else np.asarray(action, dtype=np.float32).reshape(-1)
    asset_action_offset = 1 if env.action_mode == 'cash_target_allocation' else 0
    target_weights = np.asarray(info.get('target_weights', np.zeros(len(SYMBOLS), dtype=np.float32)), dtype=np.float32).reshape(-1)
    executed_action_array = np.asarray(info.get('executed_actions', target_weights), dtype=np.float32).reshape(-1)
    record = {
        'step': step,
        'datetime': pd.to_datetime(timestamps.iloc[env.time], utc=True),
        'portfolio_value': float(env.portfolio_value),
        'benchmark_value': float(info.get('benchmark_value', env.equal_weight_value)),
        'cash': float(env.cash),
        'cash_weight': float(info.get('cash_weight', env.cash / max(env.portfolio_value, 1e-12))),
        'target_cash_weight': float(info.get('target_cash_weight', 0.0)),
        'reward': float(reward),
        'portfolio_return': float(info.get('portfolio_return', 0.0)),
        'drawdown': float(info.get('drawdown', 0.0)),
        'total_fees': float(info.get('total_fees', 0.0)),
        'turnover_penalty': float(info.get('turnover_penalty', 0.0)),
        'rolling_volatility': float(info.get('rolling_volatility', 0.0)),
        'risk_adjusted_return': float(info.get('risk_adjusted_return', 0.0)),
        'cost_penalty': float(info.get('cost_penalty', 0.0)),
        'risk_halted': bool(info.get('risk_halted', False)),
    }
    if asset_action_offset:
        record['action_cash'] = float(action_array[0]) if action_array.size else 0.0
    for index, symbol in enumerate(SYMBOLS):
        slug = symbol_slug(symbol)
        price = float(env.price_array[env.time, index])
        holding = float(env.holdings[index])
        action_index = index + asset_action_offset
        record[f'price_{slug}'] = price
        record[f'holding_{slug}'] = holding
        record[f'position_value_{slug}'] = holding * price
        record[f'action_{slug}'] = float(action_array[action_index]) if action_index < len(action_array) else 0.0
        record[f'target_weight_{slug}'] = float(target_weights[index]) if index < len(target_weights) else 0.0
        record[f'executed_action_{slug}'] = float(executed_action_array[index]) if index < len(executed_action_array) else 0.0
    return record


def evaluate_trading_model(model, bundle: dict, label: str) -> pd.DataFrame:
    env = make_trading_env(bundle)
    timestamps = pd.to_datetime(bundle['timestamps']['datetime'], utc=True)
    observation, info = env.reset()

    records = [make_eval_record(env, timestamps, step=0, reward=0.0, action=None, info=info)]
    done = False
    step = 0
    lstm_state = None
    episode_start = np.array([True], dtype=bool)

    while not done:
        if USE_LSTM:
            action, lstm_state = model.predict(
                observation,
                state=lstm_state,
                episode_start=episode_start,
                deterministic=EVALUATION_DETERMINISTIC,
            )
        else:
            action, _ = model.predict(observation, deterministic=EVALUATION_DETERMINISTIC)

        observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        step += 1
        records.append(make_eval_record(env, timestamps, step=step, reward=reward, action=action, info=info))
        episode_start = np.array([done], dtype=bool)

    result = pd.DataFrame(records)
    result['label'] = label
    return result


test_results = evaluate_trading_model(model, test_bundle, label='test_2026_0401_0428')
test_csv_path = REPORT_DIR / f'{MODEL_NAME}_test_results.csv'
test_results.to_csv(test_csv_path, index=False)
display(test_results.head())
display(test_results.tail())

## Metrics for Worthiness Check

In [ ]:
PERIODS_PER_YEAR = 365 * 24 * 12
RETURN_EPS = 1e-12


def add_curve_columns(frame: pd.DataFrame) -> pd.DataFrame:
    enriched = frame.copy()
    enriched['agent_return'] = enriched['portfolio_value'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    enriched['benchmark_return'] = enriched['benchmark_value'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    enriched['agent_cumulative_return'] = enriched['portfolio_value'] / enriched['portfolio_value'].iloc[0] - 1.0
    enriched['benchmark_cumulative_return'] = enriched['benchmark_value'] / enriched['benchmark_value'].iloc[0] - 1.0
    enriched['agent_drawdown'] = enriched['portfolio_value'] / enriched['portfolio_value'].cummax() - 1.0
    enriched['benchmark_drawdown'] = enriched['benchmark_value'] / enriched['benchmark_value'].cummax() - 1.0
    position_cols = [f'position_value_{symbol_slug(symbol)}' for symbol in SYMBOLS]
    enriched['gross_exposure'] = enriched[position_cols].sum(axis=1) / enriched['portfolio_value'].clip(lower=RETURN_EPS)
    for symbol in SYMBOLS:
        slug = symbol_slug(symbol)
        enriched[f'exposure_{slug}'] = enriched[f'position_value_{slug}'] / enriched['portfolio_value'].clip(lower=RETURN_EPS)
    return enriched


def annualized_return(values: pd.Series, timestamps: pd.Series) -> float:
    elapsed_seconds = max((timestamps.iloc[-1] - timestamps.iloc[0]).total_seconds(), 1.0)
    years = elapsed_seconds / (365 * 24 * 60 * 60)
    return float((values.iloc[-1] / max(values.iloc[0], RETURN_EPS)) ** (1 / years) - 1)


def sharpe_ratio(returns: pd.Series) -> float:
    std = returns.std(ddof=0)
    if std <= RETURN_EPS:
        return 0.0
    return float((returns.mean() / std) * math.sqrt(PERIODS_PER_YEAR))


def sortino_ratio(returns: pd.Series) -> float:
    downside = returns[returns < 0]
    downside_std = downside.std(ddof=0)
    if downside_std <= RETURN_EPS or downside.empty:
        return 0.0
    return float((returns.mean() / downside_std) * math.sqrt(PERIODS_PER_YEAR))


def summarize_strategy(frame: pd.DataFrame, value_col: str, return_col: str, drawdown_col: str) -> dict:
    returns = frame[return_col]
    values = frame[value_col]
    max_drawdown = abs(float(frame[drawdown_col].min()))
    ann_return = annualized_return(values, frame['datetime'])
    ann_vol = float(returns.std(ddof=0) * math.sqrt(PERIODS_PER_YEAR))
    return {
        'initial_value': float(values.iloc[0]),
        'final_value': float(values.iloc[-1]),
        'total_return_pct': float((values.iloc[-1] / values.iloc[0] - 1.0) * 100),
        'annualized_return_pct': ann_return * 100,
        'annualized_volatility_pct': ann_vol * 100,
        'sharpe': sharpe_ratio(returns),
        'sortino': sortino_ratio(returns),
        'max_drawdown_pct': max_drawdown * 100,
        'calmar': float(ann_return / max(max_drawdown, RETURN_EPS)),
        'win_rate_pct': float((returns > 0).mean() * 100),
    }


test_results = add_curve_columns(test_results)
test_results.to_csv(test_csv_path, index=False)

holding_cols = [f'holding_{symbol_slug(symbol)}' for symbol in SYMBOLS]
trade_events = int((test_results[holding_cols].diff().abs().sum(axis=1) > 1e-8).sum())
active_assets = int(sum((test_results[f'holding_{symbol_slug(symbol)}'].abs() > 1e-12).any() for symbol in SYMBOLS))

agent_metrics = summarize_strategy(test_results, 'portfolio_value', 'agent_return', 'agent_drawdown')
benchmark_metrics = summarize_strategy(test_results, 'benchmark_value', 'benchmark_return', 'benchmark_drawdown')
agent_metrics.update({
    'total_fees': float(test_results['total_fees'].sum()),
    'trade_events': trade_events,
    'active_assets_traded': active_assets,
    'mean_gross_exposure_pct': float(test_results['gross_exposure'].mean() * 100),
    'risk_halted': bool(test_results['risk_halted'].any()),
})
benchmark_metrics.update({
    'total_fees': 0.0,
    'trade_events': 0,
    'mean_gross_exposure_pct': 100.0,
    'risk_halted': False,
})

metrics_df = pd.DataFrame({'agent': agent_metrics, 'equal_weight_benchmark': benchmark_metrics}).T
metrics_df['alpha_vs_benchmark_pct'] = metrics_df['total_return_pct'] - metrics_df.loc['equal_weight_benchmark', 'total_return_pct']
display(metrics_df.round(4))

metrics_path = REPORT_DIR / f'{MODEL_NAME}_test_metrics.json'
metrics_payload = json.loads(metrics_df.reset_index().rename(columns={'index': 'strategy'}).to_json(orient='records'))
with metrics_path.open('w', encoding='utf-8') as handle:
    json.dump(metrics_payload, handle, indent=2)

agent = metrics_df.loc['agent']
benchmark = metrics_df.loc['equal_weight_benchmark']
passes_return   = agent['total_return_pct'] > benchmark['total_return_pct']
passes_sharpe   = agent['sharpe'] > benchmark['sharpe']
passes_drawdown = agent['max_drawdown_pct'] <= benchmark['max_drawdown_pct']
passes_positive = agent['total_return_pct'] > 0
score   = sum([passes_return, passes_sharpe, passes_drawdown, passes_positive])
verdict = 'PASS for deeper research' if score >= 3 else 'NOT WORTH DEPLOYING YET'
print(f'Worthiness check: {verdict} ({score}/4 conditions passed)')
print(f'Saved enriched test results: {test_csv_path}')

## Plots for Model Analysis

In [ ]:
sns.set_theme(style='whitegrid')

plot_frame = test_results.copy()
plot_frame['datetime'] = pd.to_datetime(plot_frame['datetime'], utc=True)
rolling_window = 12 * 24 * 7
plot_frame['agent_7d_return'] = plot_frame['portfolio_value'] / plot_frame['portfolio_value'].shift(rolling_window) - 1.0
plot_frame['benchmark_7d_return'] = plot_frame['benchmark_value'] / plot_frame['benchmark_value'].shift(rolling_window) - 1.0

fig, axes = plt.subplots(5, 1, figsize=(15, 20), sharex=True)

axes[0].plot(plot_frame['datetime'], plot_frame['portfolio_value'], label='Agent Portfolio', linewidth=1.3)
axes[0].plot(plot_frame['datetime'], plot_frame['benchmark_value'], label='Equal-Weight Benchmark', linewidth=1.1)
axes[0].set_title('Portfolio Value')
axes[0].set_ylabel('USDT')
axes[0].legend()

axes[1].plot(plot_frame['datetime'], plot_frame['agent_cumulative_return'] * 100, label='Agent', linewidth=1.2)
axes[1].plot(plot_frame['datetime'], plot_frame['benchmark_cumulative_return'] * 100, label='Benchmark', linewidth=1.2)
axes[1].set_title('Cumulative Return')
axes[1].set_ylabel('%')
axes[1].legend()

axes[2].fill_between(plot_frame['datetime'], plot_frame['agent_drawdown'] * 100, 0, alpha=0.35, label='Agent')
axes[2].plot(plot_frame['datetime'], plot_frame['benchmark_drawdown'] * 100, label='Benchmark', linewidth=1.0)
axes[2].set_title('Drawdown')
axes[2].set_ylabel('%')
axes[2].legend()

for symbol in SYMBOLS:
    slug = symbol_slug(symbol)
    axes[3].plot(plot_frame['datetime'], plot_frame[f'exposure_{slug}'] * 100, label=f'{symbol} exposure', linewidth=1.0)
axes[3].plot(plot_frame['datetime'], plot_frame['gross_exposure'] * 100, label='Gross exposure', linewidth=1.2, linestyle='--')
axes[3].set_title('Portfolio Exposure')
axes[3].set_ylabel('% of portfolio')
axes[3].legend()

for symbol in SYMBOLS:
    slug = symbol_slug(symbol)
    axes[4].plot(plot_frame['datetime'], plot_frame[f'action_{slug}'], label=f'{symbol} action', linewidth=0.8, alpha=0.8)
axes[4].axhline(0, color='black', linewidth=0.8)
axes[4].set_title('Model Actions')
axes[4].set_ylabel('Action')
axes[4].legend()

plt.tight_layout()
combined_plot_path = PLOT_DIR / f'{MODEL_NAME}_test_analysis.png'
fig.savefig(combined_plot_path, dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(plot_frame['datetime'], plot_frame['agent_7d_return'] * 100, label='Agent 7d return')
axes[0].plot(plot_frame['datetime'], plot_frame['benchmark_7d_return'] * 100, label='Benchmark 7d return')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Rolling 7-Day Return')
axes[0].set_ylabel('%')
axes[0].legend()

sns.histplot(plot_frame['agent_return'] * 100, bins=80, kde=True, ax=axes[1], label='Agent', color='tab:blue', stat='density')
sns.histplot(plot_frame['benchmark_return'] * 100, bins=80, kde=True, ax=axes[1], label='Benchmark', color='tab:orange', stat='density', alpha=0.45)
axes[1].set_title('5m Return Distribution')
axes[1].set_xlabel('Return %')
axes[1].legend()

plt.tight_layout()
risk_plot_path = PLOT_DIR / f'{MODEL_NAME}_risk_diagnostics.png'
fig.savefig(risk_plot_path, dpi=180, bbox_inches='tight')
plt.show()

## Artifact Summary

In [ ]:
artifact_summary = {
    'model_zip': str(MODEL_PATH) + '.zip',
    'training_summary': str(MODEL_DIR / f'{MODEL_NAME}_training_summary.json'),
    'test_results_csv': str(test_csv_path),
    'test_metrics_json': str(metrics_path),
    'combined_analysis_plot': str(combined_plot_path),
    'risk_diagnostics_plot': str(risk_plot_path),
    'train_processed_bundle': str(train_bundle['root']),
    'test_processed_bundle': str(test_bundle['root']),
}
display(pd.DataFrame(artifact_summary.items(), columns=['artifact', 'path']))
with (REPORT_DIR / f'{MODEL_NAME}_artifact_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_summary, handle, indent=2)
print(f"Saved artifact summary: {REPORT_DIR / f'{MODEL_NAME}_artifact_summary.json'}")
print(f'\nAll outputs saved under: {WORKING_ROOT}')
print('Use Kaggle Output tab or Save Version to persist these files.')

## Optional: Final Full-Window Retrain

Set `RUN_FINAL_FULL_WINDOW_RETRAIN = True` in the config cell (cell 4) once the worthiness check passes.

In [ ]:
if RUN_FINAL_FULL_WINDOW_RETRAIN:
    full_frames = split_symbol_frames(full_symbol_frames, DATA_START_ISO, DATA_END_ISO, inclusive_end=True)
    full_bundle, _ = build_normalized_feature_bundle(
        full_frames,
        PROCESSED_FULL_ROOT,
        DATA_START_ISO,
        DATA_END_ISO,
        label='full_2024_0101_2026_0428',
    )
    full_env = make_vec_env(full_bundle)
    final_name = f'{MODEL_NAME}_full_window'
    final_path = MODEL_DIR / final_name
    final_model_kwargs = dict(
        env=full_env,
        verbose=1,
        learning_rate=3e-5,
        n_steps=4096,
        batch_size=1024,
        gamma=0.995,
        gae_lambda=0.95,
        clip_range=0.08,
        ent_coef=0.001,
        target_kl=0.02,
        device=DEVICE,
    )
    if USE_LSTM:
        final_model = RecurrentPPO('MlpLstmPolicy', **final_model_kwargs)
    else:
        final_model = PPO(
            'MlpPolicy',
            policy_kwargs={'net_arch': [256, 256]},
            **final_model_kwargs,
        )
    final_model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
    final_model.save(final_path)
    full_env.close()
    print(f'Saved final full-window model: {final_path}.zip')
else:
    print('Skipped final full-window retrain. Set RUN_FINAL_FULL_WINDOW_RETRAIN = True after the test metrics are acceptable.')